In [1]:
from config import llm_local as llm
from config import SERPAPI_API_KEY

import os

from crewai import Agent, Task, Crew
from langchain_community.utilities import GoogleSerperAPIWrapper

#serpapi key for google search
import os
SERPAPI_API_KEY = os.getenv("SERPAPI_API_KEY")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")


In [2]:
# Configure LLM directly

from crewai import LLM

## Example for local LLM server (e.g., LM Studio)
llm = LLM(
    model="openai/lmstudio-community/gemma-3n-E4B-it-text-GGUF",  # This is just a placeholder
    base_url="http://localhost:1234/v1",
    api_key="lm-studio"
)

# FOR OPENROUTER
# llm = LLM(
#     model="openrouter/mistralai/mistral-7b-instruct:free",
#     base_url="https://openrouter.ai/api/v1",
#     api_key=OPENROUTER_API_KEY
# )



### Agent Tools

In [3]:
from crewai_tools import ScrapeWebsiteTool, SerperDevTool, FileReadTool

search_tool = SerperDevTool(delay=1, n_results=5)  # 1 second delay between requests
scrape_tool = ScrapeWebsiteTool() 
file_read_tool = FileReadTool() 

c:\Users\gathe\Documents\02_bioinfo\crewproject\.venv\Lib\site-packages\pydantic\_internal\_config.py:323: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  warnings.warn(DEPRECATION_MESSAGE, DeprecationWarning)


### Creating Agents

In [4]:
tech_news_agent = Agent(
    role="Tech News Researcher",
    goal="Collect the latest news, product releases and articles on latest research breakthroughs and findings on {topic}.",
    backstory="You are a diligent and resourceful tech news researcher. You have access to the internet and can use tools to gather information. Your goal is to find the most recent and relevant news articles, product releases, and research breakthroughs related to the {topic}. You are gathering information for a report on the latest developments in {topic}.",
    allow_delegation=True,
    tools=[search_tool, scrape_tool],
    system_message="Always provide clean, structured responses without markdown formatting.",
    llm=llm
)

In [5]:
tech_consultant_agent = Agent(
    role="Tech Consultant",
    goal="Evaluates and analyses the findings gathered by the Tech News Researcher, determines the significance and general trends and developments to highlight with regards to {topic}",
    backstory="You are an expert technologist with a background in industry and applied research and development. You are evaluating the findings gathered by the Tech News Researcher to determine the most significant trends and developments in {topic}. You will provide insights and recommendations based on the latest news, product releases, and research breakthroughs.",
    allow_delegation=True,
    tools=[search_tool, scrape_tool],
    llm=llm
)

In [6]:
tech_writer_agent = Agent(
    role="Tech Report Writer",
    goal="write a summary report of the key findings about {topic}",
    backstory="You are skilled in science communications and science writing. You are working on producing a report of no more than three pages summarizing the findings of the Tech News Researcher and the analysis and evaluation provided by Tech Consultant. Your audience are scientists and researchers interested in latest developments and products/equipment/technologies for {topic}. You will write in clear, concise language and avoid jargon. You will not carry out your own search on the topic, but rely on the findings and analysis provided by the Tech News Researcher and Tech Consultant.",
    allow_delegation=True,
    #tools=[search_tool, scrape_tool],
    #tools=[file_read_tool],
    llm=llm
)

In [7]:
tech_editor_agent = Agent(
    role="Tech Report Editor",
    goal="Ensures that the final report is in a clear and consistent format, without grammatical errors and contradictions.",
    backstory="You are skilled in editing and proofreading. You are working on ensuring that the final report is in a clear and consistent format, without grammatical errors and contradictions. You will ensure that the report is well-structured and easy to read. You will also ensure that the report is free of any inconsistencies.",
    allow_delegation=False,
    #tools=[search_tool, scrape_tool],
    #tools=[file_read_tool], 
    llm=llm
)

### Tasks

In [8]:
# task for tech researcher
# to gather relevant information on the topic


from pydantic import BaseModel
# Define a Pydantic model for search results 
# (demonstrating Output as Pydantic)
class TechDetails(BaseModel):
    author: str
    website: str
    date_published: str
    summary: str



# Task for tech news researcher:
gather_info_task = Task(
    description=(
        "Search for news, articles, product releases and research papers related to {topic} "
        "for {application} starting {from_year}. "
        "Summarize the key findings in a concise manner. "
        
    ),
    expected_output=(
        "Save all findings in clean JSON format with tech details following the exact schema provided. Do not include backticks or comments."
    ),
    output_json=TechDetails,
    output_file="tech_news_findings.json",
    agent=tech_news_agent,
    output_pydantic=None
)




In [9]:
# task for tech consultant
# to evaluate and analyse the findings of the tech researcher
# to determine the significance and general trends and developments to highlight with regards to the topic

analysis_task = Task(
    description=(
        "Evaluate and analyse the findings gathered by the Tech News Researcher on {topic}. "
        "Perform a deeper dive into the findings from the Tech News Researcher. "
        "Provide insights and recommendations for research directions, potential applications, and market opportunities."
    ),
    expected_output=(
        "A detailed analysis of the key trends and developments in {topic} for {application}, "
        "including their significance and potential impact."
    ),
    output_file="tech_analysis_report.txt",
    agent=tech_consultant_agent,
)


In [10]:
# task for tech writer
# to write a summary report of the key findings about the topic

write_report_task = Task(
    description=(
        "Write a summary based only on the findings of the Tech News Researcher and the analysis provided by Tech Consultant. "
        "Write in clear, concise language and avoid jargon. "
        
    ),
    expected_output=(
        "A well-written and concise report summarizing the key findings about {topic}. "
    ),
    context=[analysis_task, gather_info_task],  # include gather_info_task as context
    output_file="tech_summary_report.txt",
    agent=tech_writer_agent
    )



# there is some disconnect here that is apparent when we read the report generated by this agent. it is not following up on the findings and analysis of the tect consultant

In [11]:
# task for editor
# to ensure that the final report is in a clear and consistent format, without grammatical errors and contradictions

edit_report_task = Task(
    description=(
        "Ensure that the final report is in a clear and consistent format, without grammatical errors and contradictions. "
        "Ensure that the report is well-structured and easy to read. "
        "Ensure that the report is free of any factual errors or inconsistencies."
    ),
    expected_output=(
        "A polished and error-free report in markdown that is clear, consistent, and easy to read."
    ),
    # output_file="tech_final_report.md",
    agent=tech_editor_agent,
    output_file="tech_final_report.md",
)

In [12]:
from crewai import Crew, Process


# Define the crew with agents and tasks
techscan_crew = Crew(
    agents=[tech_news_agent, 
            tech_consultant_agent, 
            tech_writer_agent, 
            tech_editor_agent],
    
    tasks=[gather_info_task, 
           analysis_task, 
           write_report_task, 
           edit_report_task],
    
    manager_llm=llm,
    process=Process.hierarchical,
    max_iterations=2,
    verbose=True
)

In [13]:
# Example data for kicking off the process
techscan_inputs = {
    'topic': 'genomics',
    'application': 'cancer diagnostics',
    'from_year': '2020',
}


In [14]:
### this execution will take some time to run
result = techscan_crew.kickoff(inputs=techscan_inputs)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: d5a03881-cf52-470d-a732-8969f8a98941                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Task: Search for news, articles, product releases and research papers related to genomics for cancer           │
│  diagnostics starting 2020. Summarize the key findings in a concise manner.                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

c:\Users\gathe\Documents\02_bioinfo\crewproject\.venv\Lib\site-packages\rich\live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Thought: I need to find information about genomics for cancer diagnostics starting in 2020. I will use the     │
│  search tool to gather relevant articles, news, and research papers. I should focus on findings since 2020 and  │
│  summarize them in JSON format.                                                                                 │
│                                                                                                                 │
│  Using Tool: Search the internet with Serper                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  "{\"search_query\": \"genomics cancer diagnostics 2020\"}"                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  "{\"task\": {\"description\": \"Polish and format the final report\", \"type\": \"str\"}, \"context\":         │
│  {\"description\": \"Ensure that the final report is in a clear and consistent format, without grammatical      │
│  errors and contradictions. Ensure that the report is well-structured and easy to read. Ensure that the report  │
│  is free of any factual errors or inconsistencies.\", \"type\": \"str\"}, \"coworker\": {\"description\":       │
│  \"Tech Report Editor\", \"type\": \"str\"}}"                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│  I encountered an error while trying to use the tool. This was the error: unhashable type: 'dict'.              │
│   Tool Delegate work to coworker accepts these inputs: Tool Name: Delegate work to coworker                     │
│  Tool Arguments: {'task': {'description': 'The task to delegate', 'type': 'str'}, 'context': {'description':    │
│  'The context for the task', 'type': 'str'}, 'coworker': {'description': 'The role/name of the coworker to      │
│  delegate to', 'type': 'str'}}                                                                                  │
│  Tool Description: Delegate a specific task to one of the following coworkers: Tech Report Editor               │
│  The input to this tool should be the coworker, the task you want them to do, and ALL necessary context to      │
│  execute the task, they know nothing about the task, so share absolutely everything you know, don't reference   │
│  things but instead explain them..                                                                              │
│  Moving on then. I MUST either use a tool (use one at time) OR give my best final answer not both at the same   │
│  time. When responding, I must use the following format:                                                        │
│                                                                                                                 │
│  ```                                                                                                            │
│  Thought: you should always think about what to do                                                              │
│  Action: the action to take, should be one of [Delegate work to coworker, Ask question to coworker]             │
│  Action Input: the input to the action, dictionary enclosed in curly braces                                     │
│  Observation: the result of the action                                                                          │
│  ```                                                                                                            │
│  This Thought/Action/Action Input/Result can repeat N times. Once I know the final answer, I must return the    │
│  following format:                                                                                              │
│                                                                                                                 │
│  ```                                                                                                            │
│  Thought: I now can give a great answer                                                                         │
│  Final Answer: Your final answer must be the great and the most complete as possible, it must be outcome        │
│  described                                                                                                      │
│                                                                                                                 │
│  ```                                                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Genomics in Cancer: A Summary for Researchers                                                                │
│                                                                                                                 │
│  ## Genomics in Cancer: A Summary for Researchers                                                               │
│                                                                                                                 │
│  Genomics is revolutionizing cancer care, moving towards personalized medicine through increasingly             │
│  sophisticated diagnostic approaches. Key advancements are centered around liquid biopsies, next-generation     │
│  sequencing (NGS), and the identification of biomarkers such as microRNAs (miRNAs) and circulating tumor        │
│  antigens. These technologies facilitate earlier and more accurate diagnoses, predict patient responses to      │
│  therapies like chemotherapy and immunotherapy, and guide the selection of targeted treatments.                 │
│                                                                                                                 │
│  **Key Technological Advancements:**                                                                            │
│                                                                                                                 │
│  *   **Liquid Biopsies:** Offer a non-invasive way to analyze tumor DNA circulating in the bloodstream,         │
│  enabling real-time monitoring of disease progression and treatment response.                                   │
│  *   **Next-Generation Sequencing (NGS):** Allows for comprehensive analysis of the tumor genome, identifying   │
│  specific mutations driving cancer development and providing insights into potential therapeutic targets.       │
│  *   **Biomarker Identification:**  Discovery of biomarkers like miRNAs and circulating tumor antigens          │
│  provides valuable information about tumor characteristics and predicts treatment outcomes.                     │
│                                                                                                                 │
│  **Future Directions & Research Opportunities:**                                                                │
│                                                                                                                 │
│  The future of genomics in cancer lies in integrating multi-omics data (genomics, transcriptomics, proteomics,  │
│  etc.) and leveraging artificial intelligence (AI) to analyze complex genomic datasets.  Key research areas     │
│  include:                                                                                                       │
│                                                                                                                 │
│  *   Improving the sensitivity and specificity of liquid biopsies.                                              │
│  *   Expanding the panel of actionable biomarkers for targeted therapies.                                       │
│  *   Developing AI algorithms for advanced genomic data analysis and novel biomarker discovery.                 │
│                                                                                                                 │
│  **Market Opportunities:**                             

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 6be22b8f-b5be-42fd-b268-63b7ab49337b                                                                     │
│  Agent: Crew Manager                                                                                            │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: d5a03881-cf52-470d-a732-8969f8a98941                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: # Genomics in Cancer: A Summary for Researchers                                                  │
│                                                                                                                 │
│  ## Genomics in Cancer: A Summary for Researchers                                                               │
│                                                                                                                 │
│  Genomics is revolutionizing cancer care, moving towards personalized medicine through increasingly             │
│  sophisticated diagnostic approaches. Key advancements are centered around liquid biopsies, next-generation     │
│  sequencing (NGS), and the identification of biomarkers such as microRNAs (miRNAs) and circulating tumor        │
│  antigens. These technologies facilitate earlier and more accurate diagnoses, predict patient responses to      │
│  therapies like chemotherapy and immunotherapy, and guide the selection of targeted treatments.                 │
│                                                                                                                 │
│  **Key Technological Advancements:**                                                                            │
│                                                                                                                 │
│  *   **Liquid Biopsies:** Offer a non-invasive way to analyze tumor DNA circulating in the bloodstream,         │
│  enabling real-time monitoring of disease progression and treatment response.                                   │
│  *   **Next-Generation Sequencing (NGS):** Allows for comprehensive analysis of the tumor genome, identifying   │
│  specific mutations driving cancer development and providing insights into potential therapeutic targets.       │
│  *   **Biomarker Identification:**  Discovery of biomarkers like miRNAs and circulating tumor antigens          │
│  provides valuable information about tumor characteristics and predicts treatment outcomes.                     │
│                                                                                                                 │
│  **Future Directions & Research Opportunities:**                                                                │
│                                                                                                                 │
│  The future of genomics in cancer lies in integrating multi-omics data (genomics, transcriptomics, proteomics,  │
│  etc.) and leveraging artificial intelligence (AI) to analyze complex genomic datasets.  Key research areas     │
│  include:                                                                                                       │
│                                                                                                                 │
│  *   Improving the sensitivity and specificity of liquid biopsies.                                              │
│  *   Expanding the panel of actionable biomarkers for targeted therapies.                                       │
│  *   Developing AI algorithms for advanced genomic data analysis and novel biomarker discovery.                 │
│                                                       